In [1]:
import os
import time
import logging
from google.cloud import bigquery
from dotenv import load_dotenv
import vertexai
from google.cloud import discoveryengine_v1 as discoveryengine
from vertexai import generative_models as genai  # Añadir esta línea
from vertexai.generative_models import (
    FunctionDeclaration,
    GenerationConfig,
    Tool,
)

In [2]:
load_dotenv()  # Carga las variables desde .env al entorno
client = bigquery.Client(project='dataton-2024-team-01-cofares')
# Ahora puedes acceder a las variables de entorno
project_id = os.getenv("GOOGLE_CLOUD_PROJECT")

# Configuración de logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("app.log"),  # Archivo de log
        logging.StreamHandler()            # Consola
    ]
)

logger = logging.getLogger(__name__)

In [3]:
# Configuración del cliente de Vertex AI
PROJECT_ID = "dataton-2024-team-01-cofares"
LOCATION = "us-central1"
vertexai.init(project=PROJECT_ID, location=LOCATION)
#multimodal_model = GenerativeModel("gemini-1.5-flash-001")

# Inicializa el cliente de Discovery Engine
discovery_client = discoveryengine.RankServiceClient()  

In [4]:

def get_products(refined_query):
    client = bigquery.Client(project=project_id)
    query = """
    WITH QueryEmbedding AS (
      SELECT
        ml_generate_embedding_result AS query_embedding
      FROM
        ML.GENERATE_EMBEDDING(
          MODEL `dataton-2024-team-01-cofares.datos_cofares.text_embedding`,
          (SELECT @prompt AS content),  -- Aquí usamos el parámetro
          STRUCT(TRUE AS flatten_json_output, 'RETRIEVAL_QUERY' AS task_type)
        )
    )
    SELECT
      d.nombre_completo_material AS nombre,
      d.txt_mas_informacion_del_producto AS descripcion,
      d.txt_instrucciones_de_uso AS modo_implementacion,
      d.codigo_web,
      d.URI_primera_imagen,
      d.codigo_nacional,
      ML.DISTANCE(
        qe.query_embedding,
        e.ml_generate_embedding_result,
        'COSINE'
      ) AS distance_to_query
    FROM
      `dataton-2024-team-01-cofares.datos_cofares.data_final_temp` AS d
    INNER JOIN
      `dataton-2024-team-01-cofares.datos_cofares.SalidaEmbeddings_temp` AS e
      ON d.codigo_web = e.title
    INNER JOIN QueryEmbedding AS qe
      ON TRUE
    ORDER BY
      distance_to_query
    LIMIT 10;
    """.format(refined_query)
    # Configura el parámetro para el prompt
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("prompt", "STRING", refined_query)
        ]
    )

    query_job = client.query(query, job_config=job_config)
    results = query_job.result()
    
    products = []
    for row in results:

        descripcion = row.descripcion
        if not row.descripcion:
            descripcion = '-'
        
        modo_implementacion = row.modo_implementacion
        if not row.modo_implementacion:
            modo_implementacion = '-'


        # Cambia la URL si es necesario
        imagen_url = row.URI_primera_imagen 
        if imagen_url and imagen_url.startswith('gs:/'):
            imagen_url = imagen_url.replace('gs://dataton-2024-team-01-cofares-datastore/imagenes/', 'https://storage.googleapis.com/dataton-2024-team-01-cofares-datastore/imagenes/reto_cofares/')
        products.append({
            "codigo_web": row.codigo_web,
            "nombre": row.nombre,
            "codigo_nacional": row.codigo_nacional,
            "descripcion": descripcion,
            "modo_implementacion": modo_implementacion,
            "imagen_url": imagen_url,
            "distance_to_query": row.distance_to_query
        })
    return products

In [5]:
def rerank_products(refined_query, products):
    ranking_config = discovery_client.ranking_config_path(
        project=PROJECT_ID,
        location=LOCATION,
        ranking_config="default_ranking_config",
    )
    
    records = [
        discoveryengine.RankingRecord(
            id=str(index),
            title=product["nombre"],
            content=product["descripcion"] + " " + product["modo_implementacion"]
        )
        for index, product in enumerate(products)
    ]
    
    request = discoveryengine.RankRequest(
        ranking_config=ranking_config,
        model="semantic-ranker-512@latest",
        top_n=10, # cantidad de productos a rankear
        query=refined_query,
        records=records,
    )
    
    response = discovery_client.rank(request=request)
    
    # Aseguramos que los productos están formateados según el esquema
    ranked_products = [
        {
            "codigo_web": products[int(record.id)]["codigo_web"],
            "nombre": products[int(record.id)]["nombre"],
            "codigo_nacional": products[int(record.id)]["codigo_nacional"],
            "descripcion": products[int(record.id)]["descripcion"],
            "modo_implementacion": products[int(record.id)]["modo_implementacion"],
            "imagen_url": products[int(record.id)]["imagen_url"],
            "distance_to_query": products[int(record.id)]["distance_to_query"]
        }
        for record in response.records[:5] # cantidad de productos a mostrar
    ]
    
    return {"products": ranked_products}

In [6]:
#FUNCTION CALLING

# Define el schema
product_schema = FunctionDeclaration(
    name="product_query",
    description="Fetches relevant product information based on a search prompt.",
    parameters={
        "type": "object",
        "properties": {
            "products": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "codigo_web": {"type": "string", "description": "Product web code"},
                        "nombre": {"type": "string", "description": "Product name"},
                        "codigo_nacional": {"type": "string", "description": "National product code"},
                        "descripcion": {"type": "string", "description": "Product description"},
                        "modo_implementacion": {"type": "string", "description": "Mode of implementation"},
                        "imagen_url": {"type": "string", "description": "Image URL"},
                        "distance_to_query": {"type": "number", "description": "Semantic distance to query"}
                    }
                }
            }
        }
    }
)

# Define tools antes de inicializar el modelo
tools = [Tool(function_declarations=[product_schema])]

In [7]:
PROJECT_ID = "dataton-2024-team-01-cofares"  # @param {type:"string"}
LOCATION = "us-central1"  # @param {type:"string"}

# Importa el modelo de Gemini Flash 1.5
import vertexai
vertexai.init(project=PROJECT_ID, location=LOCATION)


# Model definition
multimodal_model = genai.GenerativeModel(
"gemini-1.5-flash",
generation_config=GenerationConfig(temperature=0),
tools=tools)

chat = multimodal_model.start_chat(response_validation=False)

# Lista para almacenar el historial de mensajes
message_history = []
# Construir el historial de la conversación
historial_conversacion = "\n".join([f"{message['role']}: {message['content']}" for message in message_history])


In [8]:
# INTENTAMOS DEVOLVER LA LISTA DE PRODUCTOS RANKED
def generate_response(prompt):
    instruction_prompt = f"""
    # Instrucción
    Eres Cofinder, un asistente farmacéutico experto.\
    Tu tarea consiste en responder eficazmente a las consultas de los profesionales de farmacia.\
    Te proporcionamos una lista de productos procedentes de la base de datos y previamente rankeados por relevancia.\
    Primero debes leer atentamente la entrada del usuario,\
    y luego desarrollar una respuesta basada en los Criterios proporcionados en la sección Producto a continuación.\
    
    # Producto
    ## Definición de la herramienta
    Tienes acceso a una lista de productos de una base de datos de productos de farmacia "{tools}"\
    que han sido reordenados para proporcionar la mejor respuesta posible a la consulta de un profesional de farmacia.\
    Las instrucciones para realizar la tarea de respuesta se proporcionan en las consultas del usuario.\
    Cuando envies el prompt al RAG reescribe la consulta para que sea más precisa y eficiente.
    
    ## Criterios
    - Si la entrada del profesional de farmacia es un saludo, preséntese cordialmente como Cofinder el asistente de búsqueda.\
        Ejemplos de saludos: «hola», “hola”, “¿Qué tal?”.\
    - Si es necesario, puede pedir detalles aclaratorios para ajustar la búsqueda a resultados eficientes.\
    - Si la entrada solicita búsquedas no relacionadas con productos de farmacia, aclare que ese no es su propósito como asistente de búsqueda de productos de farmacia.\
        Ejemplos de solicitudes no pertinentes: «Quiero la receta de una lasaña», “Quiero pedir una pizza”, “¿Qué tiempo hace hoy?”.\
    - Cuando la entrada sea relevante para activar la búsqueda de productos de farmacia, utiliza "tools" para recibir una lista de productos de farmacia clasificados que ayuden al usuario con su tarea. Acepta la solicitud del usuario y proporciónale la lista de productos sin reescribirla.
    - No sugieras ni añadas productos que no estén en la lista proporcionada por el reranker.

    ### Prompt

        Aquí está la consulta del experto farmacéutico: {prompt}
    """

    try:
        # Actualizar el historial de mensajes antes de enviar el mensaje al modelo
        message_history.append({"role": "user", "content": prompt})

        # Enviar el mensaje al modelo
        response = chat.send_message(instruction_prompt)

        # Registrar la respuesta del modelo en el historial
        message_history.append({"role": "assistant", "content": response.text})

        # Inspeccionar y loggear el historial de mensajes
        logger.info("Historial de mensajes:")
        for idx, message in enumerate(message_history):
            logger.info(f"Mensaje {idx + 1} - Role: {message['role']}, Content: {message['content']}")

        # Construir un prompt para que Gemini resuma la interacción
        resumen_prompt = """
        # Instrucción
        Eres un asistente experto en farmacia. Tu tarea es resumir la interacción con el usuario para extraer las palabras clave más relevantes para una búsqueda de productos farmacéuticos. 
        A continuación se presenta el historial de la conversación. Por favor, proporciona un resumen conciso que capture el intento del usuario y las palabras clave importantes para la búsqueda.

        # Historial de la conversación
        {}
        
        # Resumen
        """.format("\n".join([f"{message['role']}: {message['content']}" for message in message_history]))

        # Enviar el prompt de resumen al modelo
        resumen_response = chat.send_message(resumen_prompt)

        # Obtener el resumen generado por el modelo
        refined_query = resumen_response.text.strip()

        # Logging para depuración
        logger.info(f"Query refinada generada a partir del resumen:\n{refined_query}")

        # Manejo de la lógica para el RAG o conversación
        if "product_search" in response.text:
            products = get_products(refined_query)
            if not products:
                return {
                    "type": "error",
                    "message": "Lo siento, no encontré productos que coincidan con tu búsqueda."
                }

            ranked_products = rerank_products(refined_query, products)

            return {
                "type": "product_search",
                "message": "He encontrado los siguientes productos:",
                "products": ranked_products["products"]
            }
        else:
            return {
                "type": "conversation",
                "message": response.text
            }

    except Exception as e:
        return {
            "type": "error",
            "message": f"Lo siento, ocurrió un error: {str(e)}"
        }

In [11]:
# Ejemplo de uso
prompt = "hola"

In [11]:

products = get_products(prompt)  # Llamar a la función para obtener productos
# Imprimir los productos obtenidos
print("Productos obtenidos:")
for product in products:
    print(f"Nombre: {product['nombre']}, Descripción: {product['descripcion']}, Modo de implementación: {product['modo_implementacion']}, Distancia: {product['distance_to_query']}")

Productos obtenidos:
Nombre: PLANTAS MACA BIO 60CAP, Descripción: La Maca en cápsulas de Santiveri es un compuesto natural originario del Perú y es utilizado desde hace cientos de años por los pueblos incas debido a su propiedades medicinales. La maca es un tubérculo rico en sustancias nutritivas y energéticas.Entre las propiedades de la maca destacan como tratamiento de la impotencia sexual, la disfunción eréctil y para reducir los problemas provocados por la menopausia.Además es rica en minerales y fitoesteroles que ayudan a combatir la osteoporosis. También aporta energía y vitalidad ya que contiene sustancias que estimulan y regulan el organismo.La maca además mejora el transporte de oxígeno, previene la anemia y la baja presión arterial. También aumenta las defensas que ayudan a fortalecer el sistema inmunológico de manera natural gracias a sus alcaloides60 cápsulas, Modo de implementación: Tomar una cápsula tres veces al día, preferiblemente separadas de las comidas.Se recomienda

In [12]:
# Llamar a la función de reranking
ranked_products = rerank_products(prompt, products)["products"]  # Accede a la lista de productos

# Imprimir los productos rankeados
print("Productos rankeados:")
for product in ranked_products:
    print(f"Nombre: {product['nombre']}, Descripción: {product['descripcion']}, Modo de implementación: {product['modo_implementacion']}, Distancia: {product['distance_to_query']}")

Productos rankeados:
Nombre: PRANADERM RELAX CR 50ML, Descripción: PRANADERM RELAX DE 50 ML, CREMA RELAJANTE NOCTURNA, Modo de implementación: Uso tópico. Aplicar en la piel con un suave masaje, preferiblemente por la noche, antes de acostarse. - Masaje facial suave. - Corazón 7 (C7), muñeca, Shen Men oreja, timo 18VC, punto epífisis en oreja. Evitar el contacto con los ojos. En caso de hipersensibilidad, suspender la aplicación. Mantener fuera del alcance de los niños., Distancia: 0.9196770646247944
Nombre: CONTORNO OJOS PRO-COLLAG 15ML, Descripción: Potencia la producción de colágeno en piel. Hidrata, nutre y regenera. Proporciona antioxidantes naturales para combatir el paso del tiempo. Acción antimanchas y protección uva-uvb natural, que actúa en el interior de la piel gracias al ácido hialurónico natural. 15 ml, Modo de implementación: Aplicar sobre la piel limpia del contorno de los ojos y párpados realizando un suave masaje desde el centro de los ojos hacia las sienes. CONSERVAC

In [12]:
response_text = generate_response(prompt)  # Generar la respuesta
print(response_text)

2024-11-19 13:34:36,156 - INFO - Historial de mensajes:
2024-11-19 13:34:36,158 - INFO - Mensaje 1 - Role: user, Content: crema para rejuvenecimiento de la piel
2024-11-19 13:34:36,160 - INFO - Mensaje 2 - Role: user, Content: hola
2024-11-19 13:34:36,168 - INFO - Mensaje 3 - Role: assistant, Content: Hola, soy Cofinder, tu asistente de búsqueda de productos de farmacia. ¿En qué puedo ayudarte hoy? 

2024-11-19 13:34:36,713 - INFO - Query refinada generada a partir del resumen:
El usuario está buscando una crema para rejuvenecimiento de la piel.


{'type': 'conversation', 'message': 'Hola, soy Cofinder, tu asistente de búsqueda de productos de farmacia. ¿En qué puedo ayudarte hoy? \n'}
